Installing libraries and dependencies

In [34]:
from qiskit import QuantumCircuit
from qiskit import transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
import numpy as np
import qiskit
import math
from math import sqrt
import matplotlib.pyplot as plt
from qiskit.visualization import array_to_latex
from qiskit.circuit.library import RYGate

## Dividing a 2^n vector into two 2^n-1 subvectors and treating complex angles' magnitudes

While the subvector is larger than 1, it will be recursively split into two other subvectors of equal size. After the last recursive call, the probability values are computed into a new vector angles, which is returned as a vector of magnitudes


In [35]:
def gen_angles(x):
    if len(x) > 1:
        new_x = list(range(len(x)//2))

        for k in range(len(new_x)):
            new_x[k] = sqrt(x[2*k]**2 + x[2*k+1]**2)

        inner_angles = gen_angles(new_x)

        angles = list(range(len(x)//2))
        for k in range(len(angles)):
            if new_x[k] != 0:
                if x[2*k] > 0:
                    angles[k] = 2 * np.arcsin(x[2*k+1] / new_x[k])
                else:
                    angles[k] = 2 * np.pi - 2 * np.arcsin(x[2*k+1] / new_x[k])
            else:
             angles[k] = 0

        angles = inner_angles + angles
        return angles 
    else:
        return []               

## Validating the generation of the normalized magnitudes vector

In [36]:
target_angles = [sqrt(0.03), sqrt(0.07), sqrt(0.15), sqrt(0.05), sqrt(0.1), sqrt(0.3),
                 sqrt(0.2), sqrt(0.1)]

target_angles_normalized = gen_angles(target_angles)
print(target_angles_normalized)

[np.float64(1.9823131728623846), np.float64(1.9106332362490186), np.float64(1.4274487578895316), np.float64(1.9823131728623846), np.float64(1.0471975511965979), np.float64(2.0943951023931957), np.float64(1.230959417340775)]


## Preparing and normalizing complex angles' phases

In [37]:
def gen_angles_z(x):
    if len(x) > 1:
        new_x = list(range(len(x)//2))

        for k in range(0, len(new_x)):
            new_x[k] = (x[2*k] + x[2*k+1])/2

        inner_angles = gen_angles_z(new_x)

        angles_z = list(range(len(x)//2))

        for k in range(0, len(new_x)):
            angles_z[k] = x[2*k+1] - x[2*k]

        angles_z = inner_angles + angles_z
        return angles_z

    else:
        return []
    

## Validating the gen_angles_z function with a 2-qubit vector

In [38]:
phases = [ np.sqrt(0.1) * np.exp(1j * 0),
    np.sqrt(0.3) * np.exp(1j * np.pi / 4),
    np.sqrt(0.2) * np.exp(1j * np.pi / 2),
    np.sqrt(0.4) * np.exp(1j * 3 * np.pi / 4)]

target_phases = np.angle(phases)

target_phases_normalized = gen_angles_z(target_phases)

print(target_phases_normalized)

[np.float64(1.5707963267948966), np.float64(0.7853981633974483), np.float64(0.7853981633974483)]


# Implementation of gen_circuit

The `gen_circuit` function builds the state preparation quantum circuit based on the angles computed by the `gen_angles` function.

In [39]:
def gen_circuit(angles, angles_z = None):

    N = len(angles) + 1
    n = int(math.log2(N))
    circuit = QuantumCircuit(n)
    q = circuit.qubits[::-1]

    def level(k):
        return int(math.log2(k + 1))

    def index(k, j, q, circuit):
        s = k - (2**j - 1)
        for i in range(j):
            bit = (s >> (j - 1 - i)) & 1
            if bit == 0:
                circuit.x(q[i])

    for k in range(N - 1):
        j = level(k)
        
        index(k, j, q, circuit)
        angle = angles[k]
        if j == 0:
            circuit.ry(angle, q[0])

            if angles_z is not None:
                circuit.rz(angles_z[k], q[0])
        else:
            controlled_ry = RYGate(angle).control(num_ctrl_qubits=j)
            circuit.append(controlled_ry, [*q[:j], q[j]])

            if angles_z is not None:
                controlled_ry = RYGate(angles[k]).control(num_ctrl_qubits=j)
                circuit.append(controlled_ry, [*q[:j], q[j]])

            
        index(k, j, q, circuit)

    return circuit

# Testing and Validation of the gen_circuit Function

Let's instantiate the `gen_circuit` function with the angles from the 8-dimensional example (3 qubits) described in Section 2 of the paper to visually check the generated structure.

In [40]:

test_angles = [1.98, 1.91, 1.43, 1.98, 1.05, 2.09, 1.23]

qc = gen_circuit(test_angles)


print("Quantum Circuit Structure Output (N=8, 3 qubits):")
print(qc.draw(output='text'))

Quantum Circuit Structure Output (N=8, 3 qubits):
                                                        ┌──────────┐          »
q_0: ───────────────────────────────────────────────────┤ Ry(1.98) ├──────────»
                      ┌──────────┐     ┌──────────┐┌───┐└────┬─────┘┌───┐     »
q_1: ─────────────────┤ Ry(1.91) ├─────┤ Ry(1.43) ├┤ X ├─────■──────┤ X ├─────»
     ┌──────────┐┌───┐└────┬─────┘┌───┐└────┬─────┘├───┤     │      ├───┤┌───┐»
q_2: ┤ Ry(1.98) ├┤ X ├─────■──────┤ X ├─────■──────┤ X ├─────■──────┤ X ├┤ X ├»
     └──────────┘└───┘            └───┘            └───┘            └───┘└───┘»
«     ┌──────────┐     ┌──────────┐     ┌──────────┐
«q_0: ┤ Ry(1.05) ├─────┤ Ry(2.09) ├─────┤ Ry(1.23) ├
«     └────┬─────┘┌───┐└────┬─────┘┌───┐└────┬─────┘
«q_1: ─────■──────┤ X ├─────■──────┤ X ├─────■──────
«          │      ├───┤     │      └───┘     │      
«q_2: ─────■──────┤ X ├─────■────────────────■──────
«                 └───┘                             


# Circuit loading 

Writing the `main` function responsible for the `load_circuit` step

In [41]:
n = 4
circuit = QuantumCircuit(n)

for k in range(n - 2):
    RYGate()



TypeError: RYGate.__init__() missing 1 required positional argument: 'theta'